# Submitted Experiments Exploration

This notebook explores `expids.json`, which contains **submitted experiment payloads** from the ARTIQ dashboard.

## Key Difference from explist_debug.json

| File | Contains |
|------|----------|
| `explist_debug.json` | Experiment *definitions* with parameter schemas and defaults |
| `expids.json` | *Submitted* experiments with user-chosen values |

The goal is to understand the `ndscan_params` structure in submitted experiments so we can replicate dashboard functionality in our HTTP server.

## 1. Import Required Libraries

In [1]:
import json
from collections import Counter, defaultdict
from pprint import pprint

import pandas as pd

# For PyON parsing - ndscan_params is actually JSON-encoded in this file
# (not PYON like explist_debug.json), but we keep sipyco available
try:
    from sipyco import pyon
except ImportError:
    pyon = None
    print("sipyco not available - using json only")

## 2. Load and Parse expids.json

Each entry in this file represents a **submitted experiment** with a unique RID (Run ID).

In [2]:
# Load the submitted experiments
with open("expids.json", "r") as f:
    expids = json.load(f)

print(f"Total submitted experiments: {len(expids)}")
print(f"RID range: {min(int(k) for k in expids.keys())} - {max(int(k) for k in expids.keys())}")

# Show the structure of a single entry
sample_rid = list(expids.keys())[0]
sample_exp = expids[sample_rid]
print(f"\n=== Sample entry (RID {sample_rid}) ===")
print(f"Top-level keys: {list(sample_exp.keys())}")

Total submitted experiments: 97
RID range: 64615 - 64714

=== Sample entry (RID 64615) ===
Top-level keys: ['devarg_override', 'log_level', 'file', 'class_name', 'arguments', 'repo_rev']


## 3. Extract Experiment Metadata

Each submitted experiment contains:
- `file`: Path to the experiment Python file
- `class_name`: The experiment class
- `log_level`: Logging level (30 = WARNING)
- `repo_rev`: Git commit hash
- `arguments`: Dict containing regular args + `ndscan_params`
- `devarg_override`: Device argument overrides

In [3]:
# Extract metadata from all experiments
metadata = []
for rid, exp in expids.items():
    meta = {
        "rid": int(rid),
        "file": exp.get("file", ""),
        "class_name": exp.get("class_name", ""),
        "log_level": exp.get("log_level", None),
        "has_arguments": "arguments" in exp,
        "has_ndscan": "ndscan_params" in exp.get("arguments", {}),
        "num_args": len(exp.get("arguments", {})),
    }
    metadata.append(meta)

df_meta = pd.DataFrame(metadata)
print(f"Experiments with arguments: {df_meta['has_arguments'].sum()}")
print(f"Experiments with ndscan_params: {df_meta['has_ndscan'].sum()}")
print("\nMost common experiment classes:")
print(df_meta["class_name"].value_counts().head(10))

Experiments with arguments: 97
Experiments with ndscan_params: 97

Most common experiment classes:
class_name
LMTInterferometryWithDoubleLaunch                    54
DifferentialClockInterferometryWithNoiseAndSignal    16
RelockAllIJDs                                        11
DisplayAllSUServoMonitors                             4
Recentre461                                           3
DoubleLaunchFromXODT                                  3
Relock689Cavity                                       2
CentreAllTopticaModes                                 2
MonitorMaster                                         1
Relock698Cavity                                       1
Name: count, dtype: int64


## 4. Parse ndscan_params Structure

The `ndscan_params` field is a **JSON-encoded string** containing:
- `instances`: Fragment paths → parameter FQN lists
- `schemata`: Parameter schemas (type, default, spec)
- `always_shown`: Parameters marked always visible
- `overrides`: **User-submitted values** (the key difference!)
- `scan`: Scan configuration (axes, num_repeats, modes)

In [4]:
# Parse all ndscan_params
parsed_experiments = []

for rid, exp in expids.items():
    ndscan_str = exp.get("arguments", {}).get("ndscan_params", "")
    if ndscan_str:
        try:
            ndscan = json.loads(ndscan_str)
            parsed_experiments.append(
                {
                    "rid": int(rid),
                    "file": exp.get("file", ""),
                    "class_name": exp.get("class_name", ""),
                    "other_args": {k: v for k, v in exp.get("arguments", {}).items() if k != "ndscan_params"},
                    "ndscan": ndscan,
                }
            )
        except json.JSONDecodeError as e:
            print(f"Failed to parse RID {rid}: {e}")

print(f"Successfully parsed: {len(parsed_experiments)} experiments with ndscan_params")

# Show ndscan_params structure
if parsed_experiments:
    sample = parsed_experiments[0]
    print("\n=== ndscan_params keys ===")
    print(list(sample["ndscan"].keys()))

Successfully parsed: 97 experiments with ndscan_params

=== ndscan_params keys ===
['instances', 'schemata', 'always_shown', 'overrides', 'scan']


In [5]:
# Examine detailed structure of a sample ndscan_params
sample = parsed_experiments[0]
ndscan = sample["ndscan"]

print(f"Experiment: {sample['class_name']}")
print("\n=== instances (fragment paths → params) ===")
print(f"Number of instance paths: {len(ndscan['instances'])}")
for path, params in list(ndscan["instances"].items())[:3]:
    print(f"  '{path}': {len(params)} params")

print("\n=== schemata ===")
print(f"Number of parameter schemas: {len(ndscan['schemata'])}")

print("\n=== always_shown ===")
print(f"Count: {len(ndscan.get('always_shown', []))}")

print("\n=== overrides (USER VALUES!) ===")
print(f"Number of overridden parameters: {len(ndscan.get('overrides', {}))}")

print("\n=== scan configuration ===")
print(f"Keys: {list(ndscan['scan'].keys())}")

Experiment: RelockAllIJDs

=== instances (fragment paths → params) ===
Number of instance paths: 25
  '': 8 params
  'frag_relocker_blue_IJD1_controller': 6 params
  'frag_relocker_blue_IJD1_controller/frag_koheron_blue_IJD1_controller': 4 params

=== schemata ===
Number of parameter schemas: 98

=== always_shown ===
Count: 8

=== overrides (USER VALUES!) ===
Number of overridden parameters: 12

=== scan configuration ===
Keys: ['axes', 'num_repeats', 'no_axes_mode', 'randomise_order_globally', 'skip_on_persistent_transitory_error']


## 5. Analyze Parameter Overrides

The `overrides` dict is the **key difference** from schema definitions:
- Maps parameter FQN → list of override objects
- Each override has `{"path": "", "value": <actual_value>}`
- The `path` matches fragment instance path (empty for root)
- The `value` is the **actual Python value** (not a string!)

In [6]:
# Examine override structure in detail
print("=== Override Examples ===")

for exp in parsed_experiments[:5]:
    overrides = exp["ndscan"].get("overrides", {})
    if overrides:
        print(f"\nRID {exp['rid']}: {exp['class_name']}")
        for fqn, override_list in list(overrides.items())[:2]:
            short_fqn = fqn.split(".")[-1]
            for override in override_list:
                val = override["value"]
                print(f"  {short_fqn}: path='{override['path']}', value={val} ({type(val).__name__})")

=== Override Examples ===

RID 64615: RelockAllIJDs
  blue_IJD1_controller_enabled: path='', value=True (bool)
  num_points: path='', value=100.0 (float)

RID 64616: DisplayAllSUServoMonitors
  waittime: path='', value=0.1 (float)

RID 64617: RelockAllIJDs
  blue_IJD1_controller_enabled: path='', value=True (bool)
  num_points: path='', value=100.0 (float)

RID 64618: RelockAllIJDs
  blue_IJD1_controller_enabled: path='', value=True (bool)
  num_points: path='', value=100.0 (float)


In [7]:
# Analyze override value types across all experiments
override_types = Counter()
override_examples = defaultdict(list)

for exp in parsed_experiments:
    overrides = exp["ndscan"].get("overrides", {})
    schemata = exp["ndscan"].get("schemata", {})

    for fqn, override_list in overrides.items():
        schema = schemata.get(fqn, {})
        param_type = schema.get("type", "unknown")

        for override in override_list:
            value = override["value"]
            python_type = type(value).__name__
            key = f"{param_type} → {python_type}"
            override_types[key] += 1

            if len(override_examples[key]) < 2:
                override_examples[key].append(
                    {"fqn": fqn.split(".")[-1], "value": value, "default": schema.get("default")}
                )

print("Override type mappings (schema type → Python type):")
for type_key, count in override_types.most_common():
    print(f"  {type_key}: {count}")

Override type mappings (schema type → Python type):
  float → float: 3066
  bool → bool: 448
  int → float: 313


## 6. Compare Submitted vs Default Values

Key insight: Schema defaults are stored as **strings**, but override values are **native Python types**.

In [8]:
# Show examples of each type mapping with defaults
print("Examples of override formats (value vs default):")
for type_key, examples in list(override_examples.items())[:6]:
    print(f"\n{type_key}:")
    for ex in examples:
        print(f"  {ex['fqn']}: submitted={ex['value']!r} (default={ex['default']!r})")

Examples of override formats (value vs default):

bool → bool:
  blue_IJD1_controller_enabled: submitted=True (default='True')
  blue_IJD2_controller_enabled: submitted=True (default='True')

int → float:
  num_points: submitted=100.0 (default='100')
  num_points: submitted=100.0 (default='100')

float → float:
  current_waittime: submitted=0.005 (default='0.005')
  red_aom_frequency: submitted=366927000.0 (default='366927000.0')


In [9]:
# Analyze scan configurations across all experiments
scan_stats = {
    "no_axes_mode": Counter(),
    "randomise": Counter(),
    "skip_error": Counter(),
    "num_axes": Counter(),
    "num_repeats": [],
}

for exp in parsed_experiments:
    scan = exp["ndscan"].get("scan", {})
    scan_stats["no_axes_mode"][scan.get("no_axes_mode")] += 1
    scan_stats["randomise"][scan.get("randomise_order_globally")] += 1
    scan_stats["skip_error"][scan.get("skip_on_persistent_transitory_error")] += 1
    scan_stats["num_axes"][len(scan.get("axes", []))] += 1
    scan_stats["num_repeats"].append(scan.get("num_repeats", 1))

print("=== Scan Configuration Statistics ===")
print("\nno_axes_mode (what happens when no scan axes):")
for mode, count in scan_stats["no_axes_mode"].most_common():
    print(f"  '{mode}': {count}")

print("\nrandomise_order_globally:")
for val, count in scan_stats["randomise"].most_common():
    print(f"  {val}: {count}")

print("\nNumber of scan axes:")
for n_axes, count in sorted(scan_stats["num_axes"].items()):
    print(f"  {n_axes} axes: {count} experiments")

print(f"\nnum_repeats range: {min(scan_stats['num_repeats'])} - {max(scan_stats['num_repeats'])}")

=== Scan Configuration Statistics ===

no_axes_mode (what happens when no scan axes):
  'repeat': 66
  'single': 31

randomise_order_globally:
  False: 97

Number of scan axes:
  0 axes: 41 experiments
  1 axes: 55 experiments
  2 axes: 1 experiments

num_repeats range: 1 - 2147483647


In [10]:
# Examine experiments WITH scan axes
experiments_with_axes = [exp for exp in parsed_experiments if exp["ndscan"].get("scan", {}).get("axes", [])]

print(f"Experiments with scan axes: {len(experiments_with_axes)}")

# Collect axis examples
axis_examples = []
for exp in experiments_with_axes[:10]:
    for axis in exp["ndscan"]["scan"]["axes"]:
        axis_examples.append({"rid": exp["rid"], "class": exp["class_name"], "axis": axis})

print("\nAxis structure examples:")
for ex in axis_examples[:3]:
    print(f"\nRID {ex['rid']}: {ex['class']}")
    pprint(ex["axis"], width=100)

Experiments with scan axes: 56

Axis structure examples:

RID 64631: DifferentialClockInterferometryWithNoiseAndSignal
{'fqn': 'clock_interferometry_from_xodt.DifferentialClockInterferometryWithNoiseAndSignalFrag.injection_aom_static_frequency',
 'path': '',
 'range': {'num_points': 11,
           'randomise_order': False,
           'start': 364880000.0,
           'stop': 364919999.99999994},
 'type': 'linear'}

RID 64632: DifferentialClockInterferometryWithNoiseAndSignal
{'fqn': 'pyaion.fragments.default_beam_setter.SetBeamsToDefaults.frequency_clock_delivery',
 'path': 'clock_default_setter',
 'range': {'num_points': 21, 'randomise_order': False, 'start': 99712000.0, 'stop': 99772000.0},
 'type': 'linear'}

RID 64633: DifferentialClockInterferometryWithNoiseAndSignal
{'fqn': 'clock_interferometry_from_xodt.DifferentialClockInterferometryWithNoiseAndSignalFrag.phase_step',
 'path': '',
 'range': {'num_points': 21, 'randomise_order': False, 'start': 0.0, 'stop': 1.0},
 'type': 'linea

## 7. Search and Filter Experiments

Helper functions to find specific experiment configurations.

In [11]:
def find_by_class(class_pattern: str):
    """Find experiments by class name (partial match)"""
    return [e for e in parsed_experiments if class_pattern.lower() in e["class_name"].lower()]


def find_with_param_override(param_pattern: str):
    """Find experiments that override a specific parameter (partial match)"""
    results = []
    for exp in parsed_experiments:
        for fqn in exp["ndscan"].get("overrides", {}).keys():
            if param_pattern.lower() in fqn.lower():
                results.append(exp)
                break
    return results


def find_with_scan_axes():
    """Find experiments with active scan axes"""
    return [e for e in parsed_experiments if e["ndscan"].get("scan", {}).get("axes", [])]


# Example: Find interferometry experiments
interferometry = find_by_class("interferom")
print(f"Interferometry experiments: {len(interferometry)}")
for exp in interferometry[:3]:
    print(f"  RID {exp['rid']}: {exp['class_name']}")

Interferometry experiments: 70
  RID 64631: DifferentialClockInterferometryWithNoiseAndSignal
  RID 64632: DifferentialClockInterferometryWithNoiseAndSignal
  RID 64633: DifferentialClockInterferometryWithNoiseAndSignal
